# DS2002 · ACC Reality Check Starter

**Template — 2026-11-18 · Fall 2026**  

---

## ACC Reality Check: team notebook

Copy this notebook into your team repository and build the project in it. The sections follow the deliverables in the capstone PDF, so a grader can find each part where they expect it. Replace every TODO with your own work, and delete this paragraph when you are done.

Your published API lives in its own folder in the repository (for example `api/main.py`), because PythonAnywhere runs a Python file and not a notebook. Section 6 of this notebook only calls that API and draws a chart from what it returns.

### 0. Team, sources, and setup

**TODO:** list the team members, the two source URLs, and the date of each pull you used for the final results.

In [9]:
%pip install requests pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# TODO: imports and constants (URLs, the database file name, the raw data folder)

import os
import requests
import pandas as pd
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

#sources from pdf
TORVIK_URL = 'https://barttorvik.com/2027_team_results.csv'
SCOREBOARD_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard'
SUMMARY_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary'

DB_FILE = 'acc.db'
RAW_FOLDER = 'raw'
os.makedirs(RAW_FOLDER, exist_ok=True)

print('Raw files saved to :', RAW_FOLDER)


Raw files saved to : raw


### 1. Extract

**TODO:** pull the Torvik CSV and the ESPN scoreboard and summary JSON. Save every raw response to disk, and read from the saved copy when it exists so that a rerun does not call the source again.

In [11]:
# TODO: Torvik ratings snapshot

#get current date time
today = datetime.now(ZoneInfo('America/New_York')).date().isoformat()
torvik_file = os.path.join(RAW_FOLDER, f'torvik_{today}.csv')

if not os.path.exists(torvik_file):
    response = requests.get(TORVIK_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout = 30)
    response.raise_for_status()
    open(torvik_file, 'w').write(response.text)

torvik_df = pd.read_csv(torvik_file)
torvik_df.head()

,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Kansas,B12,4-0,121.523298,5,91.773206,2,0.961912,1,...,0.0,0.0,0,0,0.840974,0,1.017816,8,106,72.424413
1,2,Duke,ACC,3-2,119.560171,14,91.760073,1,0.954496,2,...,0.0,0.0,0,0,0.804536,0,-0.448018,124,352,72.017440
2,3,Houston,B12,4-0,120.395781,11,92.429770,4,0.954343,3,...,0.0,0.0,0,0,0.846144,0,0.796035,22,119,67.777347
3,4,Florida,SEC,5-0,121.812198,4,93.871934,8,0.952407,4,...,0.0,0.0,0,0,0.860756,0,1.907520,1,36,70.437981
4,5,Illinois,B10,4-1,124.317648,1,96.124388,18,0.950629,5,...,0.0,0.0,0,0,0.841033,0,0.961899,9,125,74.492327


In [12]:
# TODO: ESPN scoreboard, one request per date
import json

start = date(2026, 11, 1)
end = datetime.now(ZoneInfo('America/New_York')).date() - pd.Timedelta(days=1)

day = start
while day <= end:
    date_str = day.strftime('%Y%m%d')
    scoreboard_file = os.path.join(RAW_FOLDER, f'scoreboard_{date_str}.json')
    if not os.path.exists(scoreboard_file):
        response = requests.get(SCOREBOARD_URL, 
                                params={'groups' : 2, 'dates': date_str, 'limit': 200},
                                headers={'User-Agent': 'Mozilla/5.0'}, timeout = 30)
        response.raise_for_status()
        open(scoreboard_file, 'w').write(response.text)
    day += pd.Timedelta(days=1)

print(len(os.listdir(RAW_FOLDER)), 'raw files saved to :', RAW_FOLDER)

18 raw files saved to : raw


In [16]:
# TODO: ESPN game summaries, one request per finished game

for sb in os.listdir(RAW_FOLDER):
    if sb.startswith('scoreboard_') and sb.endswith('.json'):
        scoreboard_file = os.path.join(RAW_FOLDER, sb)
        with open(scoreboard_file) as f:
            scoreboard_data = json.load(f)
        for event in scoreboard_data['events']:
            if event['competitions'][0]['status']['type']['completed']:
                game_id = event['id']
                summary_file = os.path.join(RAW_FOLDER, f'summary_{game_id}.json')
                if not os.path.exists(summary_file):
                    response = requests.get(SUMMARY_URL, 
                                            params={'event': game_id},
                                            headers={'User-Agent': 'Mozilla/5.0'}, timeout = 30)
                    response.raise_for_status()
                    open(summary_file, 'w').write(response.text)

print(len(os.listdir(RAW_FOLDER)), 'raw files saved to :', RAW_FOLDER)


103 raw files saved to : raw


In [20]:
#Check 
files = os.listdir(RAW_FOLDER)
torvik_files = [f for f in files if f.startswith('torvik_')]
scoreboard_files = [f for f in files if f.startswith('scoreboard_')]
summary_files = [f for f in files if f.startswith('summary_')]

print(f'Torvik files: {len(torvik_files)}')
print(f'Scoreboard files: {len(scoreboard_files)}')
print(f'Summary files: {len(summary_files)}')
print("ACC teams in Torvik ratings snapshot:", (torvik_df['conf'] == 'ACC').sum())

Torvik files: 1
Scoreboard files: 17
Summary files: 85
ACC teams in Torvik ratings snapshot: 18


### 2. Transform

**TODO:** one function or clearly labeled cell per step, each printing a row count.

In [ ]:
# TODO: the team name crosswalk, with the unmatched names printed before and after

In [ ]:
# TODO: games, with the Eastern game date derived from the UTC start time

In [ ]:
# TODO: team_games, one row per team per game, with margin and possessions

In [ ]:
# TODO: player_games

### 3. Load into SQLite

**TODO:** create the tables with primary keys, load them so that a rerun cannot duplicate rows, and print the row count of every table.

In [ ]:
# TODO: schema

In [ ]:
# TODO: load, then prove that a second run leaves the counts unchanged

### 4. Questions

Each question gets a SQL query run against your database, a chart if the PDF asks for one, and an answer written in full sentences that includes the number it rests on.

#### Q1. Record versus rating

In [ ]:
# TODO: SQL

In [ ]:
# TODO: chart 1

_Q1 answer._

#### Q2. Strength of the opponents so far

In [ ]:
# TODO: SQL

In [ ]:
# TODO: chart 2

_Q2 answer._

#### Q3. Close games and recent form

In [ ]:
# TODO: SQL, including a window function for the rolling margin

In [ ]:
# TODO: chart 3

_Q3 answer._

#### Q4. Pace and points per possession

In [ ]:
# TODO: SQL

_Q4 answer._

#### Q5. Who carries each team

In [ ]:
# TODO: SQL

_Q5 answer._

#### Q6. Does the rating predict the winner?

In [ ]:
# TODO: SQL that uses the latest snapshot taken before each game

_Q6 answer, with the number of games it covers._

### 5. Decision log

**TODO:** every decision that changed the data, with the number of rows it affected.

In [ ]:
# TODO

### 6. The published API

**TODO:** the live address, the list of routes, and chart 4, drawn from JSON returned by your own API.

In [ ]:
API_URL = 'TODO'

# TODO: call your routes with requests and draw chart 4

### 7. Reflection

**TODO:** answer the five reflection questions from the PDF, with every member contributing.

_reflection here_

### 8. Presentation video

**TODO:** the link to your team's recorded presentation, which runs no more than eight minutes and includes every member. Put the same link in the README.

_video link here_